# Simulación Montecarlo del Sistema de Embarcaciones Turísticas en el Lago Titicaca

**Asignatura:** SIS230 — Modelado Sistémico y Simulación  
**Docente:** Dr. Ing. FLORES VELASQUEZ EDELFRE  
**Equipo:** Rendo Alfonte Tarqui · Yimmy Yeyson Cuyo Zamata · Cristian Alexis Loza Torres  
**Semestre:** 2026-I — Universidad Nacional del Altiplano, Puno

---

Este notebook implementa la simulación Montecarlo del Muelle Turístico de Puno.
Se modela la llegada de grupos turísticos, el registro en ventanilla, la cola de embarque
y el ciclo completo de las embarcaciones hacia las islas del Lago Titicaca.

> **Nota metodológica:** La ficha de observación sintética no corresponde a una medición
> presencial oficial. Fue construida académicamente a partir del PDF base, fuentes públicas
> (MINCETUR, SERNANP) y supuestos operativos coherentes con el sistema real.

## 1. Importación de librerías y parámetros globales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(2026)

# Parámetros del modelo base
DURACION_JORNADA      = 240   # minutos (06:00 a 10:00)
N_REPLICAS            = 1000
LAMBDA_GRUPOS_HORA    = 8
MEDIA_INTERLLEGADA    = 7.5   # 60 / 8
N_EMBARCACIONES       = 3
CAPACIDAD             = 30
UMBRAL_SALIDA         = 24
ESPERA_MAXIMA_SALIDA  = 15
TIEMPO_PREPARACION   = 10

print('Parámetros cargados correctamente.')

## 2. Ficha de observación sintética

Se construyó una ficha representativa de la ventana 09:00–10:00 del 29 de mayo de 2026,
basada en fuentes públicas y supuestos coherentes con la tasa de 8 grupos/hora.

In [ ]:
df_ficha = pd.read_csv('../data/ficha_observacion_sintetica.csv')
print('Ficha de observación sintética:')
display(df_ficha)

resumen_ficha = pd.DataFrame({
    'Indicador': [
        'Grupos observados', 'Turistas observados',
        'Interllegada promedio (min)', 'Registro promedio (min)',
        'Tamaño promedio de grupo'
    ],
    'Valor': [
        len(df_ficha),
        df_ficha['Tamano_grupo'].sum(),
        round(df_ficha['Interllegada_min'].mean(), 2),
        round(df_ficha['Registro_min'].mean(), 2),
        round(df_ficha['Tamano_grupo'].mean(), 2)
    ]
})
print('\nResumen de la ventana observada:')
display(resumen_ficha)

## 3. Generadores de variables aleatorias

In [ ]:
def generar_interllegada(rng, media=7.5):
    """Exponencial(media). Llegadas independientes, 8 grupos/hora."""
    return rng.exponential(scale=media)

def generar_tamano_grupo(rng):
    """Triangular discreta(4, 10, 18). Mínimo 1 turista."""
    return max(1, int(round(rng.triangular(left=4, mode=10, right=18))))

def generar_registro(rng):
    """Triangular(3, 5, 7). Proceso manual con variabilidad moderada."""
    return rng.triangular(left=3, mode=5, right=7)

def generar_navegacion(rng):
    """Uniforme(25, 40). Sin preferencia clara; depende del viento."""
    return rng.uniform(25, 40)

def generar_permanencia(rng):
    """Triangular(45, 90, 180). Tour con valor más probable en 90 min."""
    return rng.triangular(left=45, mode=90, right=180)

print('Generadores definidos.')

## 4. Validación de distribuciones

Se generaron 10 000 muestras por variable para comparar medias muestrales con teóricas.

In [ ]:
df_validacion = pd.read_csv('../results/validacion_distribuciones.csv')
print('Validación de distribuciones (n = 10 000, semilla 2026):')
display(df_validacion)

In [ ]:
import os

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Histogramas de validación de distribuciones (10 000 muestras)', fontsize=14)

hist_files = [
    ('../figures/hist_interllegadas.png', 'Interllegadas\nExponencial(7.5)'),
    ('../figures/hist_tamano_grupo.png',  'Tamaño de grupo\nTriangular discreta(4,10,18)'),
    ('../figures/hist_registro.png',      'Registro\nTriangular(3,5,7)'),
    ('../figures/hist_navegacion.png',    'Navegación\nUniforme(25,40)'),
    ('../figures/hist_permanencia.png',   'Permanencia\nTriangular(45,90,180)'),
]

for ax, (path, title) in zip(axes.flat, hist_files):
    img = plt.imread(path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=9)

axes.flat[-1].axis('off')  # celda vacía
plt.tight_layout()
plt.show()

## 5. Generación de grupos con ventanilla FIFO

In [ ]:
def generar_grupos(rng, duracion, lambda_grupos):
    """Genera llegadas de grupos turísticos hasta t=duracion.
    Calcula fin de registro con ventanilla única FIFO.
    """
    t = 0.0
    grupos = []
    while True:
        t += rng.exponential(scale=60.0 / lambda_grupos)
        if t > duracion:
            break
        grupos.append({
            't_llegada': t,
            'tamano':    max(1, int(round(rng.triangular(4, 10, 18)))),
            't_registro': rng.triangular(3, 5, 7),
        })
    fin_ventanilla = 0.0
    for g in grupos:
        inicio = max(g['t_llegada'], fin_ventanilla)
        g['fin_reg']     = inicio + g['t_registro']
        fin_ventanilla   = g['fin_reg']
    return grupos

# Ejemplo de una réplica
rng_demo = np.random.default_rng(2026)
grupos_demo = generar_grupos(rng_demo, 30, 8)  # primeros 30 minutos
df_demo = pd.DataFrame(grupos_demo)[['t_llegada', 'tamano', 't_registro', 'fin_reg']].round(2)
df_demo.columns = ['Llegada (min)', 'Tamaño', 'Registro (min)', 'Fin registro']
print(f'Grupos generados en primeros 30 min de demo: {len(df_demo)}')
display(df_demo)

## 6. Simulación de una jornada

### Reglas operativas
- Una embarcación sale si llega a **24 turistas (umbral)** o si el primer grupo
  lleva **15 minutos esperando**.
- Los grupos no se dividen.
- Una embarcación no puede partir después del cierre de jornada (t > 240 min).
- Turistas no atendidos = grupos en cola al cierre.
- Espera = tiempo desde llegada del grupo hasta que la embarcación parte.

In [ ]:
def simular_jornada(
    rng, duracion=240, lambda_grupos=8,
    n_embarcaciones=3, capacidad=30,
    t_prep=10, umbral=24, espera_max=15
):
    cola   = generar_grupos(rng, duracion, lambda_grupos)
    cola.sort(key=lambda g: g['fin_reg'])
    botes  = [0.0] * n_embarcaciones
    esperas, sistemas, esperas_cola = [], [], []
    viajes = 0; atendidos = 0; ocupado = 0.0

    while cola:
        bote_idx = int(np.argmin(botes))
        t_inicio = max(botes[bote_idx], cola[0]['fin_reg'])

        if t_inicio > duracion:  # jornada cerrada
            break

        t = t_inicio
        carga, cargados, i = 0, [], 0
        limite = cola[0]['fin_reg'] + espera_max

        while i < len(cola):
            g = cola[i]
            if g['fin_reg'] > max(t, limite):
                break
            t = max(t, g['fin_reg'])
            if carga + g['tamano'] <= capacidad:
                carga += g['tamano']
                cargados.append(g)
                i += 1
                if carga >= umbral:
                    break
            else:
                break

        if not cargados:
            botes[bote_idx] = cola[0]['fin_reg']
            continue

        salida = t if carga >= umbral else max(t, limite)

        if salida > duracion:  # no parte tras el cierre
            break

        ciclo_sin_prep = (
            rng.uniform(25, 40) +
            rng.triangular(45, 90, 180) +
            rng.uniform(25, 40)
        )
        ciclo_total = ciclo_sin_prep + t_prep

        # Tiempo activo dentro de la ventana de la jornada
        ocupado += min(salida + ciclo_total, duracion) - salida
        botes[bote_idx] = salida + ciclo_total
        viajes    += 1
        atendidos += carga

        for g in cargados:
            espera = salida - g['t_llegada']          # desde llegada
            esperas.append(espera)
            esperas_cola.append(max(0., salida - g['fin_reg']))
            sistemas.append(espera + ciclo_sin_prep)

        cola = cola[len(cargados):]

    no_atendidos = sum(g['tamano'] for g in cola)
    tiempo_total  = duracion * n_embarcaciones
    espera_prom   = float(np.mean(esperas)) if esperas else 0.0
    return {
        'Espera_Promedio':       espera_prom,
        'Sistema_Promedio':      float(np.mean(sistemas))       if sistemas      else 0.0,
        'Longitud_Cola':         (lambda_grupos / 60.0) * espera_prom,
        'Utilizacion':           ocupado / tiempo_total          if tiempo_total  else 0.0,
        'Porcentaje_Espera':     float(np.mean(np.array(esperas_cola) > 0)) if esperas_cola else 0.0,
        'Percentil_95_Espera':   float(np.percentile(esperas, 95))          if esperas      else 0.0,
        'Turistas_Atendidos':    atendidos,
        'Turistas_No_Atendidos': no_atendidos,
        'Viajes':                viajes,
        'Ocupacion_Promedio':    atendidos / (viajes * capacidad) if viajes else 0.0,
    }

print('Función simular_jornada definida.')

In [ ]:
# Ejemplo de una sola jornada
rng_demo = np.random.default_rng(2026)
resultado_demo = simular_jornada(rng_demo)
print('Resultado de una sola jornada (demo):')
for k, v in resultado_demo.items():
    print(f'  {k:30s}: {round(v, 3)}')

## 7. Simulación Montecarlo — Escenario base (1 000 réplicas)

In [ ]:
rng_mc = np.random.default_rng(2026)
replicas_base = [
    simular_jornada(
        rng_mc,
        duracion=DURACION_JORNADA,
        lambda_grupos=LAMBDA_GRUPOS_HORA,
        n_embarcaciones=N_EMBARCACIONES,
        capacidad=CAPACIDAD,
        t_prep=TIEMPO_PREPARACION,
        umbral=UMBRAL_SALIDA,
        espera_max=ESPERA_MAXIMA_SALIDA,
    )
    for _ in range(N_REPLICAS)
]
df_base = pd.DataFrame(replicas_base)

print('Métricas del escenario base — promedio de 1 000 réplicas:')
resumen_base = df_base.mean(numeric_only=True).round(3).reset_index()
resumen_base.columns = ['Métrica', 'Promedio']
display(resumen_base)

## 8. Escenarios alternativos E0–E4

In [ ]:
df_resultados = pd.read_csv('../results/resultados_escenarios.csv')
print('Tabla comparativa de escenarios (promedio de 1 000 réplicas):')
display(df_resultados.round(3))

## 9. Comparación de métricas clave

In [ ]:
metricas_clave = df_resultados[[
    'Escenario', 'Espera_Promedio', 'Percentil_95_Espera',
    'Utilizacion', 'Turistas_Atendidos', 'Turistas_No_Atendidos'
]].copy()
metricas_clave['Utilizacion'] = (metricas_clave['Utilizacion'] * 100).round(1)
metricas_clave = metricas_clave.rename(columns={
    'Espera_Promedio':       'Espera prom. (min)',
    'Percentil_95_Espera':   'P95 espera (min)',
    'Utilizacion':           'Utiliz. (%)',
    'Turistas_Atendidos':    'Atendidos',
    'Turistas_No_Atendidos': 'No atendidos',
})
display(metricas_clave)

## 10. Gráficos finales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Comparación de escenarios E0–E4 (1 000 réplicas Montecarlo)', fontsize=13)

plot_files = [
    ('../figures/escenarios_espera_promedio.png',    'Espera promedio'),
    ('../figures/escenarios_percentil95.png',         'Percentil 95 de espera'),
    ('../figures/escenarios_utilizacion.png',         'Utilización de embarcaciones'),
    ('../figures/escenarios_turistas_no_atendidos.png','Turistas no atendidos'),
]

for ax, (path, title) in zip(axes.flat, plot_files):
    img = plt.imread(path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()

## 11. Distribución de esperas — escenario base vs. E1

In [ ]:
# Recopilar esperas individuales para comparar E0 y E1
rng_cmp = np.random.default_rng(2026)
esperas_e0 = []
esperas_e1 = []

for _ in range(200):  # 200 réplicas para visualización rápida
    r0 = simular_jornada(rng_cmp, n_embarcaciones=3)
    esperas_e0.append(r0['Espera_Promedio'])
    r1 = simular_jornada(rng_cmp, n_embarcaciones=4)
    esperas_e1.append(r1['Espera_Promedio'])

plt.figure(figsize=(10, 4.5))
plt.hist(esperas_e0, bins=25, alpha=0.65, label='E0 Base (3 botes)',           color='steelblue')
plt.hist(esperas_e1, bins=25, alpha=0.65, label='E1 Más embarcaciones (4 botes)', color='darkorange')
plt.axvline(np.mean(esperas_e0), color='steelblue',   linestyle='--', linewidth=1.5,
            label=f'Media E0: {np.mean(esperas_e0):.1f} min')
plt.axvline(np.mean(esperas_e1), color='darkorange',  linestyle='--', linewidth=1.5,
            label=f'Media E1: {np.mean(esperas_e1):.1f} min')
plt.title('Distribución de espera promedio — E0 vs. E1 (200 réplicas)')
plt.xlabel('Espera promedio por jornada (min)')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

## 12. Conclusiones técnicas del notebook

### Hallazgos principales

1. **El sistema base es estructuralmente congestionado.** El ciclo promedio de una
   embarcación (navegación + permanencia + retorno + preparación ≈ 180 min) supera
   la capacidad de completar más de 2 viajes en la jornada de 4 horas. Esto limita
   la atención efectiva a ≈ 120 turistas sobre una demanda de ≈ 345.

2. **E2 (menor preparación) tiene impacto marginal.** El tiempo de preparación
   representa solo el 5,6% del ciclo total. Reducirlo a la mitad no permite un
   viaje adicional y la mejora en espera es < 0,1 min.

3. **E1 y E3 son las alternativas más efectivas.** Agregar una embarcación (E1)
   o aumentar la capacidad a 40 (E3) reduce turistas no atendidos en ≈ 33–35 por jornada.

4. **E4 confirma fragilidad ante alta demanda.** Con 10 grupos/hora el sistema acumula
   304 turistas no atendidos por jornada, un 35% más que en E0.

### Recomendación operativa

Si el objetivo es maximizar turistas atendidos con la menor inversión posible:
- **Prioridad: E1** — incorporar una cuarta embarcación.
- **Alternativa: E3** — reemplazar las 3 embarcaciones por unidades de mayor capacidad.
- **No recomendado: E2** — el impacto no justifica la optimización del proceso de preparación.